In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

ratings = pd.read_csv('../data/processed/ratings_clean.csv')

user_enc = LabelEncoder()
item_enc = LabelEncoder()
ratings['user_idx'] = user_enc.fit_transform(ratings['userId'])
ratings['item_idx'] = item_enc.fit_transform(ratings['movieId'])

num_users = ratings['user_idx'].nunique()
num_items = ratings['item_idx'].nunique()
print(f"Users: {num_users}, Items: {num_items}")

train, test = train_test_split(ratings, test_size=0.2, random_state=42)

X_train = [train['user_idx'].values, train['item_idx'].values]
y_train = train['rating'].values.astype(np.float32)
X_test = [test['user_idx'].values, test['item_idx'].values]
y_test = test['rating'].values.astype(np.float32)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate, Dropout
from tensorflow.keras.regularizers import l2

embedding_dim = 50

user_input = Input(shape=(1,), name='user_input')
item_input = Input(shape=(1,), name='item_input')

user_emb = Embedding(num_users, embedding_dim, embeddings_regularizer=l2(1e-6), name='user_embedding')(user_input)
item_emb = Embedding(num_items, embedding_dim, embeddings_regularizer=l2(1e-6), name='item_embedding')(item_input)

user_vec = Flatten()(user_emb)
item_vec = Flatten()(item_emb)

concat = Concatenate()([user_vec, item_vec])

dense1 = Dense(128, activation='relu')(concat)
drop1 = Dropout(0.3)(dense1)
dense2 = Dense(64, activation='relu')(drop1)
drop2 = Dropout(0.3)(dense2)
dense3 = Dense(32, activation='relu')(drop2)

output = Dense(1)(dense3)

model = Model(inputs=[user_input, item_input], outputs=output)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=64,
    verbose=1
)

# Prediksi di test set
y_pred = model.predict(X_test).flatten()
rmse = np.sqrt(np.mean((y_test - y_pred)**2))
mae = np.mean(np.abs(y_test - y_pred))
print(f'Test RMSE: {rmse:.4f}, MAE: {mae:.4f}')

In [ ]:
model.save('../models/ncf_model.h5')

import matplotlib.pyplot as plt
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.title('Loss Curve')
plt.show()

In [ ]:
user_idx = 0
all_items = np.arange(num_items)
pred_ratings = model.predict([np.full_like(all_items, user_idx), all_items], verbose=0).flatten()
top_n = 10
top_indices = np.argsort(pred_ratings)[-top_n:][::-1]
top_movie_ids = item_enc.inverse_transform(top_indices)

movies = pd.read_csv('../data/raw/movies.csv')
print(movies[movies['movieId'].isin(top_movie_ids)][['title']])